In [1]:


%pip install -Uq "unstructured[all-docs]" 
%pip install -Uq langchain_chroma 
%pip install -Uq langchain langchain-community langchain-openai 
%pip install -Uq python_dotenv



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
from typing import List

from langchain_core.messages import HumanMessage
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from dotenv import load_dotenv

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

load_dotenv()

True

In [5]:
def  partition_document(file_path):
    elements=partition_pdf(
        filename=file_path,
        strategy='hi_res',
        infer_table_structure=True,
        extract_image_block_types=['Image'],
        extract_image_block_to_payload=True
        
    )
    
    print(f'Elements len is : {len(elements)}')
    
    return elements

file_path='/home/obs/Desktop/RAG/RAG_initial/notebooks/doc/attention-is-all-you-need-1.pdf'
elements=partition_document(file_path)


No languages specified, defaulting to English.


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

Elements len is : 264


In [6]:
def create_chunks_by_title(elements):
    
    chunks=chunk_by_title(
        elements=elements,
        max_characters=3000,
        new_after_n_chars=2500,
        combine_text_under_n_chars=500
    )
    
    print(f'chunks len : {len(chunks)}')
    
    return chunks

chunks=create_chunks_by_title(elements)

chunks len : 32


In [ ]:
from typing import List

def seperate_content_types(chunk):
    content_data={
        "text":chunk.text,
        "tables":[],
        "images":[],
        "type":['text']
    }
    
    if hasattr(chunk, 'metedata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type=type(element).__name__
            
            if element_type == 'Table':
                content_data['type'].append('table')
                html_table=getattr(element.metadata, 'text_as_html', element.text)
                content_data['table'].append(html_table)
            
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['image'].append('image')
                    image_base64=element.metadata.image_base64
                    content_data['image'].append(image_base64)
    content_data['type']=list(set(content_data['type']))
    return content_data


def create_enhanced_ai_summary(text:str, tables: List[str], images: List[str]):
    
        try:
            llm=ChatOpenAI(model='gpt-4o')
            prompt_text=f"""You are creating a searchable description for document content retrieval
            CONTENT TO ANALYZE:
            TEXT CONTENT:
            {text}
            """
            
            if tables:
                prompt_text = prompt_text + "TABLES: \n"
                for i, table in enumerate(tables):
                    prompt_text = prompt_text + f"Tabel {i+1} : \n {table} \n\n"
                    
                    prompt_text =prompt_text + """ 
                    YOUR TASK:
                    Generate a comprehensive, searchable description that covers:

                    1. Key facts, numbers, and data points from text and tables
                    2. Main topics and concepts discussed  
                    3. Questions this content could answer
                    4. Visual content analysis (charts, diagrams, patterns in images)
                    5. Alternative search terms users might use

                    Make it detailed and searchable - prioritize findability over brevity.

                    SEARCHABLE DESCRIPTION:
                    
                    """
            message_content=[{"type":"text", "text":prompt_text}]
            
            for image_base64 in images:
                message_content.append({"type": "image_url", "image_url":{"url":f"data:image/jpeg;base64,{image_base64}"}})
            
            message=HumanMessage(content=message_content)
            response=llm.invoke([message])
            return response.content
                
        except Exception as e:
            print(f'Ai summary failed : {e}')
            summary=f'{text[:200]}'
            
            if table:
                summary =summary + f'containes {len(table)} table'
            if images:
                summary =summary + f'contains {len(images)} images'
                
            return summary
                



def summarise_chunks(chunks):
    print('Processing chunks with AI summary ....')
    
    langchain_document=[]
    chunks_len=len(chunks)
    
    for i, chunk in enumerate(chunks):
        print(f'\n Processing chunk {i+1}/{chunks_len}')
        
        content_data=seperate_content_types(chunk)
                
        if content_data['images'] or content_data['tables']:
            print('\n creating AI summary.......')
            
            try:
                enhanced_content=create_enhanced_ai_summary(content_data['text'], content_data['tables'], content_data['images'])
                
                print(' Ai summari is created ...')
                print(f'\n\n AI summary is : \n {enhanced_content[:300]}...')
                
                
            except Exception as e:
                print(f' Ai summary i failed : {e}')
                enhanced_content=content_data['text']
        
        else:
            
            print('\n we did not find any tables or images')
            enhanced_content=content_data['text']
        
        doc=Document(
            page_content=enhanced_content,
            metadata={
                "original_content":json.dumps({
                    "raw_text": content_data['text'],
                    "table_html":content_data['tables'],
                    "image_base64":content_data['images']
                })
            }
        )
        langchain_document.append(doc)
        
    print(f"✅ Processed {len(langchain_document)} chunks")
    return langchain_document

processed_chunks = summarise_chunks(chunks)
                

Processing chunks with AI summary ....

 Processing chunk 1/32

 we did not find any tables or images


KeyError: 'table'